In [112]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.path.abspath('../src')) # Using absolute path is safer

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [113]:
%load_ext autoreload
%autoreload 2

import polars as pl
import numpy as np
from datetime import datetime, timedelta
import os
# setup paths

RAW_FOLDER = "../data/raw/"
MASTER_PATH = "../data/raw/mind1_chaos_master.csv"

# load one of your kaggle files eg (electronics)
file_name = "D:/Mind_1_Project/data/raw/us-shein-electronics-4395.csv"
df = pl.read_csv(os.path.join(RAW_FOLDER, file_name))

# Inject the synthetic "Messy" dates
def generate_messy_date():
    if np.random.rand() < 0.1:
        return None
    dt = datetime(2026, 1, 1) + timedelta(days=np.random.randint(0, 60))
    chance = np.random.rand()
    if chance < 0.4:
        return dt.strftime("%Y-%m-%d")
    elif chance < 0.7:
        return dt.strftime("%d/%m/%Y")
    else:
        return dt.strftime("%b %d, %Y")

messy_dates = [generate_messy_date() for _ in range(df.height)]
df = df.with_columns(pl.Series("date_unfixed", messy_dates))

df.write_csv(MASTER_PATH)

print("Successfully created chaos master with {df.height} rows.")
print("Columns availble for Mind_1 to clean: ")
print(df.columns)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Successfully created chaos master with {df.height} rows.
Columns availble for Mind_1 to clean: 
['goods-title-link--jump', 'goods-title-link--jump href', 'rank-title', 'rank-sub', 'price', 'discount', 'selling_proposition', 'color-count', 'goods-title-link', 'date_unfixed']


In [114]:
import sys
sys.path.append('../src') # allows notebook to see your folders
from actions import CleaningActions
import polars as pl

#load the chaos_master
df = pl.read_csv("../data/raw/mind1_chaos_master.csv")

# test the currency stripper on the price column 
df_test = CleaningActions.strip_currency(df, "price")

# Test the data Unifier on our date_unfixed column
df_test = CleaningActions.unify_date(df_test, "date_unfixed")

# show the result 
print("-----Before---")
print(df.select(["price", "date_unfixed"]).head(5))

print("\n---After---")
print(df_test.select(["price", "date_unfixed"]).head(5))

# Check the price if it is anumber now or not 
print(f"\nPrice Column Type : {df_test['price'].dtype}")


-----Before---
shape: (5, 2)
┌───────┬──────────────┐
│ price ┆ date_unfixed │
│ ---   ┆ ---          │
│ str   ┆ str          │
╞═══════╪══════════════╡
│ $1.36 ┆ Jan 09, 2026 │
│ $2.32 ┆ 2026-02-28   │
│ $1.70 ┆ 2026-01-09   │
│ $0.38 ┆ 2026-02-05   │
│ $3.10 ┆ null         │
└───────┴──────────────┘

---After---
shape: (5, 2)
┌───────┬──────────────┐
│ price ┆ date_unfixed │
│ ---   ┆ ---          │
│ f64   ┆ date         │
╞═══════╪══════════════╡
│ 1.36  ┆ 2026-01-09   │
│ 2.32  ┆ 2026-02-28   │
│ 1.7   ┆ 2026-01-09   │
│ 0.38  ┆ 2026-02-05   │
│ 3.1   ┆ null         │
└───────┴──────────────┘

Price Column Type : Float64


In [115]:
from environment import DataCleaningEnv

# 1. Initialize the environment
env = DataCleaningEnv("../data/raw/mind1_chaos_master.csv")

# 2. See what the agent sees (State is now a list of 5 statistical/semantic features)
initial_state = env.get_state()
print("--- INITIAL STATE (Column 0) ---")
print(f"Feature Vector: {initial_state}")
# Index 2 in features is our 'Price Tag'
print(f"Is Price Concept? {'Yes' if initial_state[2] == 1.0 else 'No'}")

# 3. Take an Action: Strip currency on the current column
# Note: action_id=0 is strip_currency. We don't pass column_name anymore 
# because the environment knows which column it's on!
reward, done = env.step(action_id=0)

print("\n--- AFTER ACTION ---")
print(f"Reward Received: {reward}")
print(f"Is Dataset fully scanned? {done}")

--- INITIAL STATE (Column 0) ---
Feature Vector: [0.0, 0.0, 0.0, 0.0, 1.0]
Is Price Concept? No

--- AFTER ACTION ---
Reward Received: -20
Is Dataset fully scanned? False


In [116]:
import torch
from brain import Mind1Agent

# input_size=5 (Statistical + Semantic features)
# output_size=7 (Our 0-6 cleaning and profit actions)
agent = Mind1Agent(input_size=5, output_size=7)

print("Mind 1 Brain (Neural Network) initialized.")

Mind 1 Brain (Neural Network) initialized.


In [132]:
import torch
import random
from environment import DataCleaningEnv

# 1. Fresh Start
env = DataCleaningEnv("../data/raw/mind1_chaos_master.csv")
# agent = Mind1Agent(...) should already be defined from previous cells

# Configuration
EPISODES = 500 
epsilon = 1.0
decay_rate = 0.992
min_epsilon = 0.05
gamma = 0.99 # Discount factor for future rewards

print("Starting Master Training (Direct Learning)...")

for episode in range(EPISODES):
    state = env.reset()
    done = False
    total_reward = 0

    while not done:
        # A. Choose Action
        if random.random() < epsilon:
            action = random.randint(0, 6)
        else:
            state_t = torch.FloatTensor(state)
            with torch.no_grad():
                action = torch.argmax(agent.model(state_t)).item()
        
        # B. Execute
        current_state_t = torch.FloatTensor(state)
        reward, done = env.step(action)
        next_state = env.get_state()
        
        # C. Learning Math (The "Update" Logic)
        if next_state is not None:
            next_state_t = torch.FloatTensor(next_state)
            
            # Target = Reward + (Predict best future reward)
            with torch.no_grad():
                target = reward + gamma * torch.max(agent.model(next_state_t)).item()
            
            # Prediction = What we thought we'd get for this action
            prediction = agent.model(current_state_t)[action]
            
            # Loss and Backprop
            loss = agent.criterion(prediction, torch.tensor(target, dtype=torch.float32))
            agent.optimizer.zero_grad()
            loss.backward()
            agent.optimizer.step()
            
            state = next_state
            
        total_reward += reward
    
    # D. Decay Epsilon
    epsilon = max(min_epsilon, epsilon * decay_rate)

    if episode % 20 == 0:
        print(f"Episode {episode:03} | Reward: {total_reward:04} | Epsilon: {epsilon:.2f}")

# SAVE THE BRAIN
torch.save(agent.model.state_dict(), "../src/mind1_brain.pth")
print("\n--- Training Complete & Master Brain Saved! ---")

Starting Master Training (Direct Learning)...
Episode 000 | Reward: 1860 | Epsilon: 0.99
Episode 020 | Reward: 1270 | Epsilon: 0.84
Episode 040 | Reward: 2130 | Epsilon: 0.72
Episode 060 | Reward: 1310 | Epsilon: 0.61
Episode 080 | Reward: 1330 | Epsilon: 0.52
Episode 100 | Reward: 5790 | Epsilon: 0.44
Episode 120 | Reward: 2230 | Epsilon: 0.38
Episode 140 | Reward: 1310 | Epsilon: 0.32
Episode 160 | Reward: 1330 | Epsilon: 0.27
Episode 180 | Reward: 1310 | Epsilon: 0.23
Episode 200 | Reward: 7630 | Epsilon: 0.20
Episode 220 | Reward: 8510 | Epsilon: 0.17
Episode 240 | Reward: 1330 | Epsilon: 0.14
Episode 260 | Reward: 1330 | Epsilon: 0.12
Episode 280 | Reward: 1310 | Epsilon: 0.10
Episode 300 | Reward: 1150 | Epsilon: 0.09
Episode 320 | Reward: 1310 | Epsilon: 0.08
Episode 340 | Reward: 1330 | Epsilon: 0.06
Episode 360 | Reward: 1330 | Epsilon: 0.06
Episode 380 | Reward: 10330 | Epsilon: 0.05
Episode 400 | Reward: 1330 | Epsilon: 0.05
Episode 420 | Reward: 1290 | Epsilon: 0.05
Episode

In [134]:
# 1. Final pass: Run the trained agent one last time
env.reset()
env.get_executive_summary()

# 2. View a sample of the cleaned data
print("Sample of Cleaned Data with Net Revenue:")
print(env.df.select(["price", "discount", "net_revenue", "date_unfixed"]).head(10))


MIND 1 EXECUTIVE SUMMARY
TOTAL PROJECTED REVENUE: $2.95

--- COLUMN ATTENTION REPORT ---
Column: goods-title-link--jump CLEAN
Column: goods-title-link--jump href DIRTY (2 nulls)
Column: rank-title.......... CLEAN
Column: rank-sub............ DIRTY (2 nulls)
Column: price............... CLEAN
Column: discount............ CLEAN
Column: selling_proposition. DIRTY (2 nulls)
Column: color-count......... CLEAN
Column: goods-title-link.... DIRTY (2 nulls)
Column: date_unfixed........ CLEAN
Column: net_revenue......... CLEAN

Sample of Cleaned Data with Net Revenue:
shape: (2, 4)
┌───────┬──────────┬─────────────┬──────────────┐
│ price ┆ discount ┆ net_revenue ┆ date_unfixed │
│ ---   ┆ ---      ┆ ---         ┆ ---          │
│ f64   ┆ f64      ┆ f64         ┆ f64          │
╞═══════╪══════════╪═════════════╪══════════════╡
│ 0.9   ┆ -0.4     ┆ 1.26        ┆ 2.026021e7   │
│ 1.21  ┆ -0.4     ┆ 1.694       ┆ 272026.0     │
└───────┴──────────┴─────────────┴──────────────┘


In [140]:
import torch
import polars as pl
from environment import DataCleaningEnv

# 1. Setup
env = DataCleaningEnv("../data/raw/morpg.csv")
state = env.reset()
done = False

# 2. Load the Master Brain
# Make sure your Mind1Agent class is defined in the notebook
agent.model.load_state_dict(torch.load("../src/mind1_brain.pth"))
agent.model.eval() # Put the brain in 'Expert Mode'

print("🧠 Mind 1 Master Brain Engaged... Cleaning in progress...")

while not done:
    state_t = torch.FloatTensor(state)
    with torch.no_grad():
        action = torch.argmax(agent.model(state_t)).item()
    
    # Execute the expert's choice
    reward, done = env.step(action)
    state = env.get_state()

# 3. Save the Cleaned Result
cleaned_df = env.df
cleaned_df.write_csv("../data/processed/mind1_cleaned_final.csv")

print("✅ Data Cleaned Successfully!")
print(f"Final Revenue Generated: ${cleaned_df['net_revenue'].sum() if 'net_revenue' in cleaned_df.columns else 0:,.2f}")

C:\Users\Ideapad\AppData\Local\Temp\ipykernel_10596\3737658819.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  agent.model.load_state_dict(torch.load("../src/mind1_brai

🧠 Mind 1 Master Brain Engaged... Cleaning in progress...
✅ Data Cleaned Successfully!
Final Revenue Generated: $-21,815.50


In [139]:
import polars as pl

# 1. Load both versions
raw_df = pl.read_csv("../data/raw/mind1_chaos_master.csv")
clean_df = pl.read_csv("../data/processed/mind1_cleaned_final.csv")

def generate_report(df_raw, df_clean):
    print("="*30 + " MIND 1 VALIDATION REPORT " + "="*30)
    
    # Check 1: Null Counts
    total_nulls_raw = df_raw.null_count().sum_horizontal()[0]
    total_nulls_clean = df_clean.null_count().sum_horizontal()[0]
    
    # Check 2: Column Types (Price & Date)
    price_type_raw = df_raw["price"].dtype
    price_type_clean = df_clean["price"].dtype
    
    # Check 3: Revenue Discovery
    rev_raw = 0 # Usually 0 because it's strings
    rev_clean = df_clean["net_revenue"].sum() if "net_revenue" in df_clean.columns else 0

    print(f"📉 NULL VALUES FIXED:    {total_nulls_raw} -> {total_nulls_clean}")
    print(f"🏷️  PRICE COLUMN TYPE:   {price_type_raw} -> {price_type_clean}")
    print(f"💰 TOTAL NET REVENUE:   ${rev_clean:,.2f}")
    
    print("-" * 86)
    print("SAMPLE COMPARISON (First 5 Rows):")
    cols_to_show = ["price", "discount", "net_revenue"] if "net_revenue" in df_clean.columns else ["price"]
    print(df_clean.select(cols_to_show).head(5))
    print("="*86)

generate_report(raw_df, clean_df)

============================== MIND 1 VALIDATION REPORT ==============================
📉 NULL VALUES FIXED:    412 -> 18938
🏷️  PRICE COLUMN TYPE:   String -> String
💰 TOTAL NET REVENUE:   $126,851.09
--------------------------------------------------------------------------------------
SAMPLE COMPARISON (First 5 Rows):
shape: (5, 3)
┌────────┬──────────┬─────────────┐
│ price  ┆ discount ┆ net_revenue │
│ ---    ┆ ---      ┆ ---         │
│ str    ┆ str      ┆ f64         │
╞════════╪══════════╪═════════════╡
│ $2.03  ┆ -22%     ┆ 1.5834      │
│ $6.48  ┆ -20%     ┆ 5.184       │
│ $1.80  ┆ null     ┆ 1.8         │
│ $0.88  ┆ -72%     ┆ 0.2464      │
│ $12.06 ┆ -40%     ┆ 7.236       │
└────────┴──────────┴─────────────┘
